# isitfair hardening: `diabetes130`

Step-by-step audit of the `diabetes130` prediction table built by
`hardening/datasets/registry.py` (see `hardening/datasets/CARDS.md`). The
audited model is the logistic-regression column `y_proba_lr`; every stage is
its own cell, timed with `perf_counter`. Warnings and exceptions are recorded
per stage in `RECORD` and written to `hardening/notebooks/results/`.

Environment knobs: `HARDEN_B` (bootstrap replicates, default 50) and
`HARDEN_STRICT` (`1` makes the last cell fail on any unexpected error).

In [1]:
import faulthandler
import json
import os
import sys
import time
import traceback
import warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "hardening" / "datasets" / "registry.py").exists():
        REPO = _c
        break
sys.path.insert(0, str(REPO / "hardening" / "datasets"))
# Inside a Jupyter kernel sys.stderr has no file descriptor; give the watchdog a real file.
_FH = open(REPO / "hardening" / "logs" / "faulthandler_diabetes130.log", "w")
faulthandler.dump_traceback_later(600, repeat=True, file=_FH)
import registry  # noqa: E402

import isitfair  # noqa: E402
from isitfair import FairnessAudit, GroupRecalibration, WassersteinPostprocessing, report_to_pdf  # noqa: E402

NAME = "diabetes130"
B = int(os.environ.get("HARDEN_B", "50"))
STRICT = os.environ.get("HARDEN_STRICT", "0") == "1"
# Bootstrap workers for discrimination/calibration; results do not depend on it.
NJOBS = int(os.environ["HARDEN_NJOBS"]) if os.environ.get("HARDEN_NJOBS") else None
SEED = 42
OUT = REPO / "hardening" / "notebooks" / "results"
REPORTS = REPO / "hardening" / "reports"
OUT.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
META = registry.meta(NAME)
THRESHOLD = META["threshold"]
RECORD = {"dataset": NAME, "B": B, "isitfair": isitfair.__version__,
          "isitfair_file": isitfair.__file__, "n_jobs": NJOBS, "stages": {}, "variants": {}}
print(f"isitfair {isitfair.__version__} from {isitfair.__file__}")
print(f"{NAME}: B={B} threshold={THRESHOLD} strict={STRICT} n_jobs={NJOBS}")


@contextmanager
def stage(name, expect_error=False):
    """Time a stage; record its warnings and any exception, then carry on."""
    entry = {"seconds": None, "warnings": [], "error": None, "expected_error": expect_error}
    RECORD["stages"][name] = entry
    t0 = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            yield entry
        except Exception as exc:  # recorded, not raised, so later stages still run
            entry["error"] = f"{type(exc).__name__}: {exc}"
            entry["traceback"] = traceback.format_exc()
            print(f"!! {name} raised {entry['error']}")
    entry["seconds"] = round(time.perf_counter() - t0, 3)
    seen = {}
    for w in caught:
        key = f"{w.category.__name__}: {str(w.message)[:300]}"
        seen[key] = seen.get(key, 0) + 1
    entry["warnings"] = [f"{k} (x{v})" for k, v in seen.items()]
    print(f"[{name}] {entry['seconds']:.2f} s, {len(caught)} warning(s)"
          + (f", ERROR {entry['error']}" if entry["error"] else ""))
    for k in entry["warnings"][:8]:
        print("   ", k)
    (OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))

isitfair 0.4.0 from C:\Users\hugoa\Desktop\isitfair\src\isitfair\__init__.py
diabetes130: B=50 threshold=0.15 strict=True n_jobs=-1


## 1. Load and validate inputs

In [2]:
with stage("load_validate"):
    df = registry.load(NAME)
    ATTRS = registry.ATTRS[NAME]
    checks = {
        "n": len(df),
        "index_is_range": isinstance(df.index, pd.RangeIndex),
        "y_true_dtype": str(df["y_true"].dtype),
        "y_true_values": sorted(df["y_true"].dropna().unique().tolist()),
        "y_true_missing": int(df["y_true"].isna().sum()),
        "y_proba_range": [float(df["y_proba_lr"].min()), float(df["y_proba_lr"].max())],
        "y_proba_missing": int(df["y_proba_lr"].isna().sum()),
        "attr_dtypes": {a: str(df[a].dtype) for a in ATTRS},
        "attr_missing": {a: int(df[a].isna().sum()) for a in ATTRS},
        "attr_levels": {a: int(df[a].nunique()) for a in ATTRS},
    }
    RECORD["checks"] = checks
    print(json.dumps(checks, indent=1))

{
 "n": 30530,
 "index_is_range": true,
 "y_true_dtype": "int64",
 "y_true_values": [
  0,
  1
 ],
 "y_true_missing": 0,
 "y_proba_range": [
  0.0104052461389193,
  0.9540901744232978
 ],
 "y_proba_missing": 0,
 "attr_dtypes": {
  "race": "object",
  "gender": "object",
  "age": "object"
 },
 "attr_missing": {
  "race": 676,
  "gender": 0,
  "age": 0
 },
 "attr_levels": {
  "race": 5,
  "gender": 3,
  "age": 10
 }
}
[load_validate] 0.04 s, 0 warning(s)


A missing subgroup label must be rejected with an actionable message. If any
attribute has missing labels, try the raw column first (recorded), then
recode to an explicit `"Unknown"` category for the rest of the notebook.

In [3]:
missing_attrs = [a for a in ATTRS if df[a].isna().any()]
if missing_attrs:
    with stage("raw_missing_labels", expect_error=True):
        FairnessAudit(y_true=df["y_true"], y_proba=df["y_proba_lr"],
                      sensitive_features={a: df[a] for a in ATTRS},
                      threshold=THRESHOLD, n_bootstrap=B, random_state=SEED, n_jobs=NJOBS)
    for a in missing_attrs:
        df[a] = df[a].fillna("Unknown")
print("recoded to 'Unknown':", missing_attrs or "none")

!! raw_missing_labels raised ValueError: `sensitive_features['race']` has 676 missing value(s). Subgroup labels cannot be missing: drop those rows from every input, or recode them as an explicit category, e.g. `df[col].fillna('unknown')`.
[raw_missing_labels] 0.01 s, 0 warning(s), ERROR ValueError: `sensitive_features['race']` has 676 missing value(s). Subgroup labels cannot be missing: drop those rows from every input, or recode them as an explicit category, e.g. `df[col].fillna('unknown')`.
recoded to 'Unknown': ['race']


In [4]:
y = df["y_true"].to_numpy(dtype=float)
p = df["y_proba_lr"].to_numpy(dtype=float)
marginals = {a: df[a].to_numpy(dtype=object) for a in ATTRS}
INTERSECTION = tuple(ATTRS[:2]) if len(ATTRS) >= 2 else None
sf = dict(marginals)
if INTERSECTION:
    sf[INTERSECTION] = None
print("intersection:", INTERSECTION)

intersection: ('race', 'gender')


## 2. Construct the audit

In [5]:
with stage("construct"):
    audit = FairnessAudit(y_true=y, y_proba=p, sensitive_features=sf, threshold=THRESHOLD,
                          n_bootstrap=B, random_state=SEED, n_jobs=NJOBS)
    print(audit)

FairnessAudit(n=30530, events=3407, threshold=0.15, n_bootstrap=50, random_state=42, attributes: race (6), gender (3), age (10), race × gender (13))
[construct] 0.15 s, 0 warning(s)


## 3. Discrimination

In [6]:
with stage("discrimination"):
    disc = audit.discrimination()
    print(disc[disc.subgroup != "GAP"][["attribute", "subgroup", "n", "n_events", "auroc",
                                         "auroc_ci_low", "auroc_ci_high"]].to_string(index=False))

    attribute                  subgroup       n  n_events    auroc  auroc_ci_low  auroc_ci_high
         race                   Overall 30530.0    3407.0 0.640601      0.632735       0.650480
         race           AfricanAmerican  5770.0     643.0 0.645942      0.626715       0.667501
         race                     Asian   174.0      16.0 0.655854           NaN            NaN
         race                 Caucasian 22893.0    2578.0 0.637921      0.628636       0.645867
         race                  Hispanic   594.0      71.0 0.708911      0.657886       0.770547
         race                     Other   423.0      36.0 0.619365      0.504262       0.714434
         race                   Unknown   676.0      63.0 0.612056      0.531817       0.687872
       gender                   Overall 30530.0    3407.0 0.640601      0.632735       0.650480
       gender                    Female 16427.0    1875.0 0.646475      0.632654       0.657645
       gender                      Male 

## 4. Calibration

In [7]:
with stage("calibration"):
    cal = audit.calibration()
    print(cal[cal.subgroup != "GAP"][["attribute", "subgroup", "n", "calibration_intercept",
                                       "calibration_slope", "brier", "ece", "ici"]].to_string(index=False))

    attribute                  subgroup       n  calibration_intercept  calibration_slope    brier      ece      ici
         race                   Overall 30530.0              -0.002150           0.819376 0.096200 0.008693 0.007437
         race           AfricanAmerican  5770.0              -0.038229           0.779296 0.096489 0.016045 0.008591
         race                     Asian   174.0              -0.137664           1.171590 0.077360 0.054489 0.030892
         race                 Caucasian 22893.0               0.008149           0.817734 0.096981 0.009202 0.007832
         race                  Hispanic   594.0               0.136046           1.274506 0.095907 0.029908 0.020562
         race                     Other   423.0              -0.193264           0.823706 0.076323 0.035700 0.018786
         race                   Unknown   676.0              -0.034817           0.606238 0.084828 0.038322 0.021940
       gender                   Overall 30530.0              -0.

## 5. Calibration curves

In [8]:
with stage("calibration_curves"):
    curves = audit.calibration_curves()
    print(curves.shape)

(3600, 7)
[calibration_curves] 147.74 s, 0 warning(s)


## 6. ROC curves

In [9]:
with stage("roc_curves"):
    roc = audit.roc_curves()
    print(roc.shape)

(3600, 7)
[roc_curves] 3.10 s, 0 warning(s)


## 7. Decision curve

In [10]:
with stage("decision_curve"):
    dc = audit.decision_curve()
    print(dc[dc.subgroup == "GAP"][["attribute", "gap_value", "gap_ci_low", "gap_ci_high"]].to_string(index=False))

    attribute  gap_value  gap_ci_low  gap_ci_high
         race   0.043697         NaN          NaN
       gender   0.128585         NaN          NaN
          age   0.127636         NaN          NaN
race × gender   0.159729         NaN          NaN
[decision_curve] 5.86 s, 0 warning(s)


## 8. Estimability

In [11]:
with stage("estimability"):
    est = audit.estimability()
    print(f"{int(est.not_estimable_final.sum())} of {len(est)} cell-metrics not reportable")

132 of 424 cell-metrics not reportable
[estimability] 0.04 s, 0 warning(s)


## 9. Hosmer-Lemeshow

In [12]:
with stage("hosmer_lemeshow"):
    hl = audit.hosmer_lemeshow()
    print(hl[hl.subgroup != "GAP"][["attribute", "subgroup", "n", "hl_statistic", "hl_df",
                                     "hl_p_value", "n_bins_used"]].to_string(index=False))

    attribute                  subgroup       n  hl_statistic  hl_df  hl_p_value  n_bins_used
         race                   Overall 30530.0     39.345758   10.0    0.000022         10.0
         race           AfricanAmerican  5770.0     21.173975   10.0    0.019913         10.0
         race                     Asian   174.0     10.910820   10.0    0.364508         10.0
         race                 Caucasian 22893.0     28.889399   10.0    0.001298         10.0
         race                  Hispanic   594.0     10.408946   10.0    0.405377         10.0
         race                     Other   423.0      8.274712   10.0    0.602023         10.0
         race                   Unknown   676.0     15.514399   10.0    0.114403         10.0
       gender                   Overall 30530.0     39.345758   10.0    0.000022         10.0
       gender                    Female 16427.0     25.601999   10.0    0.004314         10.0
       gender                      Male 14102.0     27.31091

## 10. Cell diagnostics for one intersection

In [13]:
with stage("cell_diagnostics"):
    if INTERSECTION:
        cells = audit.cell_diagnostics()
        print(cells.to_string(index=False))
    else:
        print("single attribute; no intersection to diagnose")

    attribute                  subgroup     n  n_events  shrunk  marginal_rate  cell_rate  shrinkage_factor
race × gender  AfricanAmerican / Female  3532       410   False       0.111438   0.116082          0.194297
race × gender    AfricanAmerican / Male  2238       233   False       0.111438   0.104111          0.275670
race × gender            Asian / Female    92         7    True       0.091954   0.076087          0.902517
race × gender              Asian / Male    82         9    True       0.091954   0.109756          0.912182
race × gender        Caucasian / Female 11935      1376   False       0.112611   0.115291          0.066612
race × gender          Caucasian / Male 10958      1202   False       0.112611   0.109692          0.072123
race × gender         Hispanic / Female   312        32    True       0.119529   0.102564          0.731901
race × gender           Hispanic / Male   282        39    True       0.119529   0.138298          0.751268
race × gender            Oth

## 11. Recalibration ladder on the attribute with the largest intercept gap

In [14]:
gaps = cal[(cal.subgroup == "GAP") & (cal.gap_metric == "max_intercept_gap")
           & cal.attribute.isin(ATTRS)]
RANKED = [str(a) for a in gaps.sort_values("gap_value", ascending=False).attribute]


def zero_event_levels(attr):
    ev = pd.Series(y).groupby(marginals[attr]).sum()
    return [str(k) for k, v in ev.items() if v == 0]


LADDER_BY = RANKED[0]
ZERO = zero_event_levels(LADDER_BY)
print("largest intercept gap:", LADDER_BY, "| zero-event levels:", ZERO or "none")
# A level without events cannot be recalibrated; isitfair refuses (by design).
with stage("recalibration_ladder", expect_error=bool(ZERO)):
    ladder = audit.recalibration_ladder(by=LADDER_BY)
    print(ladder.to_string(index=False)[:3000])

largest intercept gap: age | zero-event levels: ['[0-10)']
!! recalibration_ladder raised ValueError: stratum '[0-10)' of 'age' has zero events in the whole audit, so no fold can fit a group-specific calibrator for it; fewer folds will not help. Merge it into another level or drop those rows.
[recalibration_ladder] 0.01 s, 0 warning(s), ERROR ValueError: stratum '[0-10)' of 'age' has zero events in the whole audit, so no fold can fit a group-specific calibrator for it; fewer folds will not help. Merge it into another level or drop those rows.


In [15]:
FALLBACK = next((a for a in RANKED[1:] if not zero_event_levels(a)), None) if ZERO else None
if FALLBACK:
    with stage("recalibration_ladder_fallback"):
        ladder = audit.recalibration_ladder(by=FALLBACK)
        print("by", FALLBACK)
        print(ladder.to_string(index=False)[:3000])
else:
    print("no fallback needed" if not ZERO else "no attribute without zero-event levels")

by race
          rung attribute        subgroup       n  n_events  calibration_intercept  intercept_ci_low  intercept_ci_high  calibration_slope  slope_ci_low  slope_ci_high    brier  brier_ci_low  brier_ci_high      ece      ici  net_benefit        gap_metric  gap_value  event_floor below_floor not_estimable_final
          none      race         Overall 30530.0    3407.0              -0.002150         -0.035695           0.026392           0.819376      0.766056       0.881687 0.096200      0.093928       0.098396 0.008693 0.007437     0.011252               NaN        NaN         25.0       False               False
          none      race AfricanAmerican  5770.0     643.0              -0.038229         -0.124403           0.053724           0.779296      0.690199       0.883163 0.096489      0.091128       0.102308 0.016045 0.008591     0.012540               NaN        NaN         25.0       False               False
          none      race           Asian   174.0      16.0    

## 12. Mitigation on a held-out calibration split

support2 uses the committed calibration partition. Every other dataset splits
its audited test half 50/50 (stratified on the outcome, `random_state=42`):
the calibrators are fitted on one half and the before/after audit runs on the
other, so no row is scored by a calibrator fitted on it.

In [16]:
with stage("mitigation_split"):
    if NAME == "support2":
        cal_df = pd.read_csv(registry.PRED / "support2_calib.csv", dtype={a: object for a in ATTRS})
        eval_df = df
    else:
        from sklearn.model_selection import train_test_split
        i_cal, i_eval = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED,
                                         stratify=df["y_true"])
        cal_df, eval_df = df.iloc[i_cal].reset_index(drop=True), df.iloc[i_eval].reset_index(drop=True)
    for a in ATTRS:
        cal_df[a] = cal_df[a].fillna("Unknown")
    # mitigate() rejects audited levels the calibration half never saw (diabetes130:
    # gender "Unknown/Invalid"). Drop those audited rows and say so; a no-op elsewhere.
    unseen = np.zeros(len(eval_df), bool)
    for a in ATTRS:
        unseen |= ~eval_df[a].fillna("Unknown").isin(set(cal_df[a]))
    if unseen.any():
        print(f"dropping {int(unseen.sum())} audited rows with levels absent from the calibration half")
        eval_df = eval_df[~unseen].reset_index(drop=True)
    calib_data = (cal_df["y_true"].to_numpy(float), cal_df["y_proba_lr"].to_numpy(float),
                  {a: cal_df[a].to_numpy(object) for a in ATTRS})
    sf_eval = {a: eval_df[a].to_numpy(object) for a in ATTRS}
    if INTERSECTION:
        sf_eval[INTERSECTION] = None
    audit_eval = audit if NAME == "support2" else FairnessAudit(
        y_true=eval_df["y_true"].to_numpy(float), y_proba=eval_df["y_proba_lr"].to_numpy(float),
        sensitive_features=sf_eval, threshold=THRESHOLD, n_bootstrap=B, random_state=SEED, n_jobs=NJOBS)
    print(f"calibration n={len(cal_df)}, evaluation n={len(eval_df)}")

dropping 1 audited rows with levels absent from the calibration half
calibration n=15265, evaluation n=15264
[mitigation_split] 0.08 s, 0 warning(s)


In [17]:
with stage("mitigate_group_recalibration"):
    mitigated_gr = audit_eval.mitigate(method=GroupRecalibration(method="isotonic"),
                                       calibration_data=calib_data)
    a0 = ATTRS[0]
    keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
    before, after = audit_eval.calibration(), mitigated_gr[a0].calibration()
    print("BEFORE", a0); print(before[before.attribute == a0][keep].to_string(index=False))
    print("AFTER (group isotonic)"); print(after[after.attribute == a0][keep].to_string(index=False))

BEFORE race
       subgroup  calibration_intercept  calibration_slope    brier      ece
        Overall              -0.001528           0.789849 0.096337 0.011320
AfricanAmerican              -0.044023           0.813347 0.094199 0.017285
          Asian              -0.814003           0.099986 0.052287 0.078907
      Caucasian               0.018193           0.766074 0.098300 0.012270
       Hispanic               0.100511           1.411816 0.091441 0.045640
          Other              -0.722051           0.836755 0.051416 0.059477
        Unknown               0.059725           0.812896 0.090744 0.053805
            GAP                    NaN                NaN      NaN      NaN
            GAP                    NaN                NaN      NaN      NaN
AFTER (group isotonic)
       subgroup  calibration_intercept  calibration_slope    brier      ece
        Overall              -0.001047           0.587090 0.096440 0.011267
AfricanAmerican              -0.018067           0.80

In [18]:
with stage("mitigate_wasserstein"):
    mitigated_w = audit_eval.mitigate(method=WassersteinPostprocessing(random_state=SEED),
                                      calibration_data=calib_data)
    d_after = mitigated_w[a0].discrimination()
    print(d_after[(d_after.attribute == a0) & (d_after.subgroup == "GAP")][
        ["gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]].to_string(index=False))

                  gap_metric  gap_value  gap_ci_low  gap_ci_high
equal_opportunity_difference   0.218750         NaN          NaN
         predictive_equality   0.075092         NaN          NaN
          equalized_odds_gap   0.218750         NaN          NaN
          statistical_parity   0.050498         NaN          NaN
[mitigate_wasserstein] 1.87 s, 0 warning(s)


## 13. Report: HTML, then PDF

In [19]:
REPORT = REPORTS / f"{NAME}_B{B}.html"
with stage("report_html"):
    audit_eval.report(out=REPORT, title=f"{NAME}: hardening audit (B={B})",
                      metadata={"Dataset": NAME, "Model": "LogisticRegression (registry)",
                                "Threshold": str(THRESHOLD), "Bootstrap": f"B={B}, random_state={SEED}"},
                      mitigation_audits=mitigated_gr)
    print(REPORT, REPORT.stat().st_size, "bytes")

C:\Users\hugoa\Desktop\isitfair\hardening\reports\diabetes130_B50.html 2670823 bytes
[report_html] 185.21 s, 0 warning(s)


In [20]:
with stage("report_pdf"):
    pdf_path = report_to_pdf(REPORT)
    print(pdf_path, pdf_path.stat().st_size, "bytes")

C:\Users\hugoa\Desktop\isitfair\hardening\reports\diabetes130_B50.pdf 716335 bytes
[report_pdf] 18.13 s, 0 warning(s)


## 14. Input-type variants

The same audit (B = 20 to keep this cheap) with inputs passed in the forms
real pipelines produce. Each variant is classified as `raises`, `warns`,
`ok-identical` (results equal the plain-numpy baseline exactly), or
`ok-DIFFERENT` (runs silently but gives different numbers: a silent
misbehaviour unless the input itself changed, as in the ties variant).

In [21]:
VB = 20


def run_variant(name, y_in, p_in, sf_in, compare=True, ref=None):
    out = {"status": None, "warnings": [], "error": None}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            a = FairnessAudit(y_true=y_in, y_proba=p_in, sensitive_features=sf_in,
                              threshold=THRESHOLD, n_bootstrap=VB, random_state=SEED, n_jobs=NJOBS)
            frames = {"discrimination": a.discrimination(), "calibration": a.calibration(),
                      "hosmer_lemeshow": a.hosmer_lemeshow(), "decision_curve": a.decision_curve()}
        except Exception as exc:
            out["error"] = f"{type(exc).__name__}: {exc}"
            frames = None
    out["warnings"] = sorted({f"{w.category.__name__}: {str(w.message)[:200]}" for w in caught})
    if frames is None:
        out["status"] = "raises"
    else:
        num_bad = {k: int(np.isinf(f.select_dtypes("number").to_numpy(float)).sum()) for k, f in frames.items()}
        out["inf_values"] = num_bad
        if ref is not None and compare:
            same = all(frames[k].equals(ref[k]) for k in frames)
            out["identical_to_baseline"] = same
            out["status"] = ("warns, " if out["warnings"] else "") + ("ok-identical" if same else "ok-DIFFERENT")
            if not same:
                diffs = {}
                for k in frames:
                    fa, fb = frames[k], ref[k]
                    if fa.shape == fb.shape:
                        na = fa.select_dtypes("number").to_numpy(float)
                        nb = fb.select_dtypes("number").to_numpy(float)
                        diffs[k] = float(np.nanmax(np.abs(na - nb))) if na.size else 0.0
                    else:
                        diffs[k] = f"shape {fa.shape} vs {fb.shape}"
                out["max_abs_diff"] = diffs
        else:
            out["status"] = ("warns, " if out["warnings"] else "") + "ok"
    RECORD["variants"][name] = out
    print(f"{name:<28} {out['status']}" + (f"  {out['error']}" if out["error"] else ""))
    for w in out["warnings"][:5]:
        print("     ", w)
    (OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))
    return frames


with stage("variants"):
    base = run_variant("baseline_numpy", y, p, sf, compare=False)
    s_attr = {a: pd.Series(v, dtype="category") for a, v in marginals.items()}
    run_variant("category_attributes", y, p, {**s_attr, **({INTERSECTION: None} if INTERSECTION else {})}, ref=base)
    run_variant("Int64_outcome", pd.Series(y.astype(int), dtype="Int64"), p, sf, ref=base)
    run_variant("bool_outcome", y.astype(bool), p, sf, ref=base)
    run_variant("bool_outcome_series", pd.Series(y.astype(bool)), p, sf, ref=base)
    perm = np.random.default_rng(SEED).permutation(len(y))
    idx = pd.Index(perm + 10_000)
    run_variant("shuffled_index_consistent",
                pd.Series(y, index=idx), pd.Series(p, index=idx),
                {**{a: pd.Series(v, index=idx) for a, v in marginals.items()},
                 **({INTERSECTION: None} if INTERSECTION else {})}, ref=base)
    # y sorted/shuffled with its labels, p left in the original order: the
    # classic silent misalignment, which isitfair is documented to reject.
    run_variant("shuffled_index_misaligned",
                pd.Series(y).iloc[perm], pd.Series(p), sf, ref=base)
    p_ties = np.round(p, 2)
    k = max(2, len(p) // 50)  # ~2% of rows exactly 0 or 1, never all of them
    pos = np.random.default_rng(SEED).choice(len(p), size=k, replace=False)
    p_ties[pos[: k // 2]] = 0.0
    p_ties[pos[k // 2 :]] = 1.0
    run_variant("ties_and_exact_0_1", y, p_ties, sf, compare=False)

baseline_numpy               ok


category_attributes          ok-identical


Int64_outcome                ok-identical


bool_outcome                 ok-identical


bool_outcome_series          ok-identical


shuffled_index_consistent    ok-identical
shuffled_index_misaligned    raises  ValueError: `y_proba` and `y_true` are pandas objects with the same index labels in a different order. FairnessAudit aligns inputs by position, not by index, so the rows would be mismatched. Reorder them consistently, e.g. `s.loc[other.index]`, or pass `.to_numpy()` arrays built from one DataFrame.


ties_and_exact_0_1           ok
[variants] 1448.28 s, 0 warning(s)


## 15. Summary

In [22]:
unexpected = {k: v["error"] for k, v in RECORD["stages"].items()
              if v["error"] and not v["expected_error"]}
RECORD["unexpected_errors"] = unexpected
RECORD["total_seconds"] = round(sum(v["seconds"] or 0 for v in RECORD["stages"].values()), 2)
(OUT / f"{NAME}_B{B}.json").write_text(json.dumps(RECORD, indent=2, default=str))
print(pd.DataFrame({k: {"seconds": v["seconds"], "n_warnings": len(v["warnings"]),
                        "error": v["error"]} for k, v in RECORD["stages"].items()}).T.to_string())
print("unexpected errors:", unexpected or "none")
if STRICT:
    assert not unexpected, unexpected

                                seconds n_warnings                                                                                                                                                                                                              error
load_validate                     0.039        0.0                                                                                                                                                                                                                NaN
raw_missing_labels                0.009          0  ValueError: `sensitive_features['race']` has 676 missing value(s). Subgroup labels cannot be missing: drop those rows from every input, or recode them as an explicit category, e.g. `df[col].fillna('unknown')`.
construct                         0.153        0.0                                                                                                                                                                    